# 2.2. Подбор источников данных

**Цель этапа:** подобрать открытые источники данных по требованиям п.3 методических указаний:
- не менее 1 реляционного набора данных;
- не менее 1 иерархического набора данных (XML);
- не менее 1 источника с API (JSON);
- не менее 1 источника неструктурированных данных (тексты).

**Требования к каждому источнику:**
- ≥3 категориальных аспекта;
- ≥3 числовых характеристики;
- желательно — характеристика времени.

**Выбранные источники:**
1. UCI Online Retail II — реляционный (XLSX)
2. ЕМИСС/Росстат — иерархический (XML/SDMX)
3. Walmart Sales — JSON (локальный файл)
4. Amazon Reviews — неструктурированный (CSV, тексты)

## Источник 1. UCI Online Retail II (реляционный)

**Тип:** реляционный набор данных
**Формат:** XLSX (оригинал)
**Источник:** UCI Machine Learning Repository
**Ссылка:** https://archive.ics.uci.edu/dataset/502/online+retail+ii
**Период:** 01.12.2010 — 09.12.2011
**Объём:** 541 910 записей, 8 колонок

**Поля и аналитические шкалы:**

| Поле | Аналитическая шкала | Обоснование |
|------|---------------------|-------------|
| Invoice | Номинальная (ID) | Уникальный номер заказа, биекция |
| StockCode | Номинальная (ID) | Код товара, высокая уникальность |
| Description | Контентная | Текстовое описание, слабоструктурированные данные |
| Quantity | Абсолютная | Счётная величина, фиксированная единица |
| InvoiceDate | Интервальная | Дата и время, осмыслен шаг |
| Price | Отношений | Естественный ноль, допустима смена валюты |
| Customer ID | Номинальная (ID) | Идентификатор клиента, высокая уникальность |
| Country | Категориальная | Конечное множество категорий без порядка |

**Аналитическая роль:** основа для анализа потребительских предпочтений и поведенческих паттернов во времени.

In [15]:
# Импорт библиотек
import pandas as pd
import numpy as np
import requests
import time
import xml.etree.ElementTree as ET
import json

# Настройки отображения DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [16]:
# Загрузка реляционного источника (XLSX)
start = time.time()

# Оригинал в формате XLSX
df_retail = pd.read_excel(
    'online_retail_II.xlsx',
    sheet_name='Year 2010-2011',
    engine='openpyxl'
)

load_time_retail = time.time() - start

print(f"Время загрузки: {load_time_retail:.2f} сек")
print(f"Размер: {df_retail.shape}")
print(f"Колонки: {df_retail.columns.tolist()}")
print(f"\nПервые 5 строк:")
print(df_retail.head())
print(f"\nТипы данных:")
print(df_retail.dtypes)
print(f"\nПропущенные значения:")
print(df_retail.isnull().sum())
print("Общая информация")
df_retail.info()
print("Описательная статистика")
print(df_retail.describe())
print("\nОписательная статистика (категориальные):")
print(df_retail.describe(include='object'))

Время загрузки: 87.68 сек
Размер: (541910, 8)
Колонки: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Первые 5 строк:
  Invoice StockCode                          Description  Quantity         InvoiceDate  Price  Customer ID         Country
0  536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6 2010-12-01 08:26:00   2.55      17850.0  United Kingdom
1  536365     71053                  WHITE METAL LANTERN         6 2010-12-01 08:26:00   3.39      17850.0  United Kingdom
2  536365    84406B       CREAM CUPID HEARTS COAT HANGER         8 2010-12-01 08:26:00   2.75      17850.0  United Kingdom
3  536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6 2010-12-01 08:26:00   3.39      17850.0  United Kingdom
4  536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6 2010-12-01 08:26:00   3.39      17850.0  United Kingdom

Типы данных:
Invoice                object
StockCode              object
Description   

In [17]:
# Категориальные аспекты
print("Категориальные аспекты:")
print(f"  Country: {df_retail['Country'].nunique()} уникальных")
print(f"  StockCode: {df_retail['StockCode'].nunique()} уникальных")
print(f"  Description: {df_retail['Description'].nunique()} уникальных")

# Числовые характеристики
print("\nЧисловые характеристики:")
print(f"  Quantity: min={df_retail['Quantity'].min()}, max={df_retail['Quantity'].max()}")
print(f"  Price: min={df_retail['Price'].min()}, max={df_retail['Price'].max()}")
print(f"  Customer ID: {df_retail['Customer ID'].nunique()} уникальных")

# Временная характеристика
print("\nВременная характеристика:")
print(f"  InvoiceDate: {df_retail['InvoiceDate'].min()} — {df_retail['InvoiceDate'].max()}")

Категориальные аспекты:
  Country: 38 уникальных
  StockCode: 4070 уникальных
  Description: 4223 уникальных

Числовые характеристики:
  Quantity: min=-80995, max=80995
  Price: min=-11062.06, max=38970.0
  Customer ID: 4372 уникальных

Временная характеристика:
  InvoiceDate: 2010-12-01 08:26:00 — 2011-12-09 12:50:00


In [18]:
# Проверка дубликатов
duplicates = df_retail.duplicated().sum()
print(f"Дубликатов в Retail: {duplicates}")

Дубликатов в Retail: 5268


### Вывод:
источник загружен успешно. Содержит 3 категориальных аспекта (Country, StockCode, Description) и 3 числовых характеристики (Quantity, Price, Customer ID). Присутствует временная характеристика (InvoiceDate). Источник уникален.

## Источник 2. ЕМИСС/Росстат (иерархический, XML)

**Тип:** иерархический набор данных
**Формат:** XML (SDMX)
**Источник:** Единая межведомственная информационно-статистическая система (fedstat.ru)
**Показатель:** Оборот розничной торговли пищевыми продуктами, включая напитки, и табачными изделиями, проданных на розничных рынках и ярмарках (ID 43271)
**Период:** 2010, 2011
**Объём:** ~376 записей

**Структура XML:**
- CodeLists → справочник ОКАТО (103 региона)
- Description → метаданные показателя
- DataSet → серии наблюдений

**Поля и аналитические шкалы:**

| Поле | Аналитическая шкала | Обоснование |
|------|---------------------|-------------|
| okato | Номинальная (ID) | Код региона, биекция |
| region | Категориальная | Название региона, конечное множество |
| period | Категориальная | Период (январь-июнь / январь-декабрь) |
| year | Интервальная | Год, осмыслен шаг |
| value | Отношений | Значение оборота, естественный ноль |
| unit | Категориальная | Единица измерения |

**Аналитическая роль:** данные о розничном товарообороте по регионам РФ — фактор внешней среды.

**Структура XML (исходный вид):**

```xml
<generic:Series>
  <generic:SeriesKey>
    <generic:Value concept="s_OKATO" value="643"/>
  </generic:SeriesKey>
  <generic:Attributes>
    <generic:Value concept="EI" value="тысяча рублей"/>
    <generic:Value concept="PERIOD" value="январь-июнь"/>
  </generic:Attributes>
  <generic:Obs>
    <generic:Time>2010</generic:Time>
    <generic:ObsValue value="344794428,7"/>
  </generic:Obs>
</generic:Series>

In [19]:
# Парсинг XML (ЕМИСС/Росстат)
start = time.time()

tree = ET.parse('data.xml')
root = tree.getroot()

# Пространства имён SDMX
ns = {
    'generic': 'http://www.SDMX.org/resources/SDMXML/schemas/v1_0/generic',
    'structure': 'http://www.SDMX.org/resources/SDMXML/schemas/v1_0/structure',
    'common': 'http://www.SDMX.org/resources/SDMXML/schemas/v1_0/common'
}

print(f"Корневой тег: {root.tag}")
print(f"Время парсинга XML: {time.time() - start:.2f} сек")

Корневой тег: {http://www.SDMX.org/resources/SDMXML/schemas/v1_0/message}GenericData
Время парсинга XML: 0.13 сек


In [20]:
# Извлечение справочника регионов (ОКАТО)
okato_dict = {}

codelist = root.find('.//structure:CodeList[@id="s_OKATO"]', ns)
if codelist is not None:
    for code in codelist.findall('structure:Code', ns):
        value = code.get('value')
        desc = code.find('structure:Description', ns)
        if desc is not None:
            okato_dict[value] = desc.text

print(f"Регионов в справочнике: {len(okato_dict)}")
print(f"\nПримеры:")
for k, v in list(okato_dict.items())[:5]:
    print(f"  {k}: {v}")

Регионов в справочнике: 103

Примеры:
  643: Российская Федерация
  643004.АГ: Российская Федерация без учета новых субъектов (с 01.01.2023)
  030: Центральный федеральный округ
  14000000000: Белгородская область
  15000000000: Брянская область


In [21]:
# Извлечение данных из XML
records = []

for series in root.findall('.//generic:Series', ns):
    # Ключ серии (регион)
    series_key = series.find('generic:SeriesKey', ns)
    okato_code = None
    if series_key is not None:
        value_elem = series_key.find('generic:Value', ns)
        if value_elem is not None:
            okato_code = value_elem.get('value')
    
    # Атрибуты (единица измерения, период)
    period = None
    unit = None
    attrs = series.find('generic:Attributes', ns)
    if attrs is not None:
        for val in attrs.findall('generic:Value', ns):
            if val.get('concept') == 'PERIOD':
                period = val.get('value')
            elif val.get('concept') == 'EI':
                unit = val.get('value')
    
    # Наблюдения (год + значение)
    for obs in series.findall('generic:Obs', ns):
        time_elem = obs.find('generic:Time', ns)
        value_elem = obs.find('generic:ObsValue', ns)
        
        records.append({
            'okato': okato_code,
            'region': okato_dict.get(okato_code, 'Неизвестно'),
            'period': period,
            'year': time_elem.text if time_elem is not None else None,
            'value': float(value_elem.get('value').replace(',', '.')) if value_elem is not None else None,
            'unit': unit
        })

df_xml = pd.DataFrame(records)

print(f"Извлечено записей: {len(df_xml)}")
print(f"Колонки: {df_xml.columns.tolist()}")
print(df_xml.head(10))

Извлечено записей: 368
Колонки: ['okato', 'region', 'period', 'year', 'value', 'unit']
         okato                         region          period  year        value           unit
0          643           Российская Федерация     январь-июнь  2010  344794428.7  тысяча рублей
1          643           Российская Федерация  январь-декабрь  2010  764074818.1  тысяча рублей
2          643           Российская Федерация     январь-июнь  2011  374712559.6  тысяча рублей
3          643           Российская Федерация  январь-декабрь  2011  805377862.2  тысяча рублей
4          030  Центральный федеральный округ     январь-июнь  2010   97788012.3  тысяча рублей
5          030  Центральный федеральный округ  январь-декабрь  2010  211207119.0  тысяча рублей
6          030  Центральный федеральный округ     январь-июнь  2011  110557646.4  тысяча рублей
7          030  Центральный федеральный округ  январь-декабрь  2011  229628753.8  тысяча рублей
8  14000000000           Белгородская область    

In [22]:
# Характеристики XML-источника
print("Характеристики XML-Источника (ЕМИСС/Росстат)")

print("\nКатегориальные аспекты (>=3):")
print(f"  1. region: {df_xml['region'].nunique()} уникальных")
print(f"  2. period: {df_xml['period'].nunique()} уникальных ({df_xml['period'].unique().tolist()})")
print(f"  3. year: {df_xml['year'].nunique()} уникальных")
print(f"  4. unit: {df_xml['unit'].nunique()} уникальных")

print("\nЧисловые характеристики (>=3):")
print(f"  1. value: min={df_xml['value'].min()}, max={df_xml['value'].max()}, mean={df_xml['value'].mean():.2f}")
print(f"  2. okato (код): {df_xml['okato'].nunique()} уникальных")
print(f"  3. Наблюдений на регион: {df_xml.groupby('region').size().mean():.1f}")

print("\nВременная характеристика:")
print(f"  year: {df_xml['year'].min()} — {df_xml['year'].max()}")

print(f"\nОбщий размер: {df_xml.shape}")

# Общая информация
print("\nОбщая информация:")
df_xml.info()

# Описательная статистика
print("\nОписательная статистика:")
print(df_xml.describe())

print("\nОписательная статистика (категориальные):")
print(df_xml.describe(include='object'))

Характеристики XML-Источника (ЕМИСС/Росстат)

Категориальные аспекты (>=3):
  1. region: 92 уникальных
  2. period: 2 уникальных (['январь-июнь', 'январь-декабрь'])
  3. year: 2 уникальных
  4. unit: 1 уникальных

Числовые характеристики (>=3):
  1. value: min=0.0, max=805377862.2, mean=18819490.65
  2. okato (код): 92 уникальных
  3. Наблюдений на регион: 4.0

Временная характеристика:
  year: 2010 — 2011

Общий размер: (368, 6)

Общая информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 368 entries, 0 to 367
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   okato   368 non-null    object 
 1   region  368 non-null    object 
 2   period  368 non-null    object 
 3   year    368 non-null    object 
 4   value   368 non-null    float64
 5   unit    368 non-null    object 
dtypes: float64(1), object(5)
memory usage: 17.4+ KB

Описательная статистика:
              value
count  3.680000e+02
mean   1.881949e+07
std    6

### Вывод: 
источник загружен успешно. Содержит 4 категориальных аспекта (region, period, year, unit) и 3 числовых характеристики (value, okato, количество наблюдений). Временной диапазон: 2010, 2011. Источник пересекается с Retail по оси «Время».

## Источник 3. Walmart Sales (JSON)

**Тип:** источник с данными в формате JSON
**Источник:** Walmart Sales Dataset, Kaggle
**Ссылка:** https://www.kaggle.com/datasets/yasserh/walmart-dataset
**Формат:** JSON (конвертация из CSV)
**Период:** 05.02.2010 — 26.10.2012
**Объём:** 6435 записей

**Поля и аналитические шкалы:**

| Поле | Аналитическая шкала | Обоснование |
|------|---------------------|-------------|
| Store | Категориальная | Номер магазина, конечное множество |
| Month | Категориальная | Месяц продаж (1–12), конечное множество |
| Weekly_Sales | Отношений | Продажи за неделю, естественный ноль |
| Holiday_Flag | Дихотомическая | Праздничная неделя (0/1), ровно 2 состояния |
| Temperature | Отношений | Температура, естественный ноль |
| Fuel_Price | Отношений | Цена топлива, естественный ноль |
| CPI | Отношений | Индекс потребительских цен |
| Unemployment | Отношений | Уровень безработицы |

**Аналитическая роль:** макро-фактор розничных продаж США за 2010–2012. Источник пересекается с Retail по оси «Время» (2010–2011) и даёт контекст для анализа сезонности.

**Структура JSON (исходный вид):**

```json
[
  {
    "Store": 1,
    "Date": "05-02-2010",
    "Weekly_Sales": 1643690.86,
    "Holiday_Flag": 0,
    "Temperature": 42.31,
    "Fuel_Price": 2.572,
    "CPI": 211.096358,
    "Unemployment": 8.106
  },
  {
    "Store": 1,
    "Date": "12-02-2010",
    "Weekly_Sales": 1641957.44,
    "Holiday_Flag": 1,
    "Temperature": 38.51,
    "Fuel_Price": 2.548,
    "CPI": 211.242170,
    "Unemployment": 8.106
  }
]

In [23]:
start = time.time()

# 1. Чтение исходного CSV
df_walmart_csv = pd.read_csv('Walmart.csv')

print(f"CSV загружен: {len(df_walmart_csv)} записей")
print(f"Колонки: {df_walmart_csv.columns.tolist()}")

# 2. Конвертация в JSON
df_walmart_csv.to_json('walmart.json', orient='records', indent=2)

# 3. Чтение JSON обратно (имитация API-источника)
with open('walmart.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df_api = pd.json_normalize(data)

load_time_api = time.time() - start

print(f"\nВремя конвертации и загрузки JSON: {load_time_api:.2f} сек")
print(f"Записей: {len(df_api)}")
print(f"Колонки: {df_api.columns.tolist()}")
print(df_api.head())

CSV загружен: 6435 записей
Колонки: ['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

Время конвертации и загрузки JSON: 0.14 сек
Записей: 6435
Колонки: ['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
   Store        Date  Weekly_Sales  Holiday_Flag  Temperature  Fuel_Price         CPI  Unemployment
0      1  05-02-2010    1643690.90             0        42.31       2.572  211.096358         8.106
1      1  12-02-2010    1641957.44             1        38.51       2.548  211.242170         8.106
2      1  19-02-2010    1611968.17             0        39.93       2.514  211.289143         8.106
3      1  26-02-2010    1409727.59             0        46.63       2.561  211.319643         8.106
4      1  05-03-2010    1554806.68             0        46.50       2.625  211.350143         8.106


In [24]:
df_api['Store'] = df_api['Store'].astype(str)
df_api['Month'] = pd.to_datetime(df_api['Date'], format='%d-%m-%Y').dt.month

print("Категориальные аспекты:")
print(f"  1. Store: {df_api['Store'].nunique()} уникальных")
print(f"  2. Holiday_Flag: {df_api['Holiday_Flag'].nunique()} уникальных")
print(f"  3. Month: {df_api['Month'].nunique()} уникальных")

print("\nЧисловые характеристики:")
print(f"  1. Weekly_Sales: min={df_api['Weekly_Sales'].min()}, max={df_api['Weekly_Sales'].max()}")
print(f"  2. Temperature: min={df_api['Temperature'].min()}, max={df_api['Temperature'].max()}")
print(f"  3. Fuel_Price: min={df_api['Fuel_Price'].min()}, max={df_api['Fuel_Price'].max()}")
print(f"  4. CPI: min={df_api['CPI'].min()}, max={df_api['CPI'].max()}")
print(f"  5. Unemployment: min={df_api['Unemployment'].min()}, max={df_api['Unemployment'].max()}")

print("\nВременная характеристика:")
df_api['Date'] = pd.to_datetime(df_api['Date'], format='%d-%m-%Y')
print(f"  Date: {df_api['Date'].min()} — {df_api['Date'].max()}")

print("\nОбщая информация:")
df_api.info()

print("\nОписательная статистика:")
print(df_api.describe())

print("\nОписательная статистика (категориальные):")
print(df_api.describe(include='object'))

Категориальные аспекты:
  1. Store: 45 уникальных
  2. Holiday_Flag: 2 уникальных
  3. Month: 12 уникальных

Числовые характеристики:
  1. Weekly_Sales: min=209986.25, max=3818686.45
  2. Temperature: min=-2.06, max=100.14
  3. Fuel_Price: min=2.472, max=4.468
  4. CPI: min=126.064, max=227.2328068
  5. Unemployment: min=3.879, max=14.313

Временная характеристика:
  Date: 2010-02-05 00:00:00 — 2012-10-26 00:00:00

Общая информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Store         6435 non-null   object        
 1   Date          6435 non-null   datetime64[ns]
 2   Weekly_Sales  6435 non-null   float64       
 3   Holiday_Flag  6435 non-null   int64         
 4   Temperature   6435 non-null   float64       
 5   Fuel_Price    6435 non-null   float64       
 6   CPI           6435 non-null   float64       
 7   Un

### Вывод:
источник загружен успешно из локального JSON-файла. Содержит 3 категориальных аспекта (Store, Holiday_Flag, Month) и 5 числовых характеристик (Weekly_Sales, Temperature, Fuel_Price, CPI, Unemployment). Присутствует временная характеристика (Date). Источник уникален по содержанию (розничные продажи США, 2010–2012) и пересекается с Retail по оси «Время».

## Источник 4. Amazon Reviews (неструктурированный, CSV)

**Тип:** неструктурированные данные (тексты)
**Формат:** CSV
**Источник:** Kaggle
**Ссылка:** https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
**Объём:** 568 454 записи

**Поля и аналитические шкалы:**

| Поле | Аналитическая шкала | Обоснование |
|------|---------------------|-------------|
| Id | Номинальная (ID) | Идентификатор записи, биекция |
| ProductId | Номинальная (ID) | Идентификатор товара |
| UserId | Номинальная (ID) | Идентификатор пользователя |
| ProfileName | Контентная | Имя пользователя, текст |
| HelpfulnessNumerator | Абсолютная | Счётная величина, фиксированная единица |
| HelpfulnessDenominator | Абсолютная | Счётная величина, фиксированная единица |
| Score | Порядковая | Оценка 1-5, упорядоченные градации |
| Time | Интервальная | Время отзыва, осмыслен шаг |
| Summary | Контентная | Краткое резюме, текст |
| Text | Контентная | Текст отзыва, слабоструктурированные данные |

**Аналитическая роль:** тексты отзывов для анализа потребительских предпочтений и тональности.

In [25]:
# Загрузка неструктурированного источника (Amazon Reviews)
start = time.time()

df_reviews = pd.read_csv('Reviews.csv')

load_time_reviews = time.time() - start

print(f"Время загрузки: {load_time_reviews:.2f} сек")
print(f"Размер: {df_reviews.shape}")
print(f"Колонки: {df_reviews.columns.tolist()}")
print(df_reviews.head())

print("Общая информация")
df_reviews.info()

print("Описательная статистика")
print(df_reviews.describe())

print("\nОписательная статистика (категориальные):")
print(df_reviews.describe(include='object'))

Время загрузки: 5.55 сек
Размер: (568454, 10)
Колонки: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']
   Id   ProductId          UserId                      ProfileName  HelpfulnessNumerator  HelpfulnessDenominator  Score        Time                Summary  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian                     1                       1      5  1303862400  Good Quality Dog Food   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa                     0                       0      1  1346976000      Not as Advertised   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"                     1                       1      4  1219017600  "Delight" says it all   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl                     3                       3      2  1307923200         Cough Medicine   
4   5  B006K2ZZ7K  A1UQR

In [26]:
print("Категориальные аспекты:")
print(f"  ProductId: {df_reviews['ProductId'].nunique()} уникальных")
print(f"  UserId: {df_reviews['UserId'].nunique()} уникальных")
print(f"  ProfileName: {df_reviews['ProfileName'].nunique()} уникальных")
print(f"  Score: {df_reviews['Score'].nunique()} уникальных значений")

print("\nЧисловые характеристики:")
print(f"  Score: min={df_reviews['Score'].min()}, max={df_reviews['Score'].max()}, mean={df_reviews['Score'].mean():.2f}")
print(f"  HelpfulnessNumerator: min={df_reviews['HelpfulnessNumerator'].min()}, max={df_reviews['HelpfulnessNumerator'].max()}")
print(f"  HelpfulnessDenominator: min={df_reviews['HelpfulnessDenominator'].min()}, max={df_reviews['HelpfulnessDenominator'].max()}")

print("\nВременная характеристика:")
df_reviews['review_date'] = pd.to_datetime(df_reviews['Time'], unit='s')
print(f"  Диапазон: {df_reviews['review_date'].min()} — {df_reviews['review_date'].max()}")

print("\nНеструктурированные данные:")
df_reviews['text_length'] = df_reviews['Text'].astype(str).str.len()
df_reviews['summary_length'] = df_reviews['Summary'].astype(str).str.len()
print(f"  Text: средняя длина {df_reviews['text_length'].mean():.1f} символов")
print(f"  Summary: средняя длина {df_reviews['summary_length'].mean():.1f} символов")

print("\nПропущенные значения:")
print(df_reviews.isnull().sum())

Категориальные аспекты:
  ProductId: 74258 уникальных
  UserId: 256059 уникальных
  ProfileName: 218415 уникальных
  Score: 5 уникальных значений

Числовые характеристики:
  Score: min=1, max=5, mean=4.18
  HelpfulnessNumerator: min=0, max=866
  HelpfulnessDenominator: min=0, max=923

Временная характеристика:
  Диапазон: 1999-10-08 00:00:00 — 2012-10-26 00:00:00

Неструктурированные данные:
  Text: средняя длина 436.2 символов
  Summary: средняя длина 23.4 символов

Пропущенные значения:
Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
review_date                0
text_length                0
summary_length             0
dtype: int64


### Вывод:
источник загружен успешно. Содержит 4 категориальных аспекта (ProductId, UserId, ProfileName, Score) и 3 числовых характеристики (Score, HelpfulnessNumerator, HelpfulnessDenominator). Присутствует временная характеристика (Time). Неструктурированные данные: Text, Summary. Источник уникален.

In [27]:
# Проверка пересечения по времени между Retail и Walmart
retail_years = set(df_retail['InvoiceDate'].dt.year.unique())
walmart_years = set(pd.to_datetime(df_api['Date']).dt.year.unique())

common_years = retail_years & walmart_years
print(f"Общие годы для Retail и Walmart: {sorted(common_years)}")

Общие годы для Retail и Walmart: [2010, 2011]


## Сводная таблица источников

| № | Источник | Тип | Формат | Записей | Катег. аспектов | Числовых | Временной диапазон |
|---|----------|-----|--------|---------|-----------------|----------|-------------------|
| 1 | UCI Online Retail II | Реляционный | XLSX | 541 910 | 3 | 3 | 01.12.2010 — 09.12.2011 |
| 2 | ЕМИСС/Росстат | Иерархический | XML | 368 | 4 | 3 | 2010, 2011 |
| 3 | Walmart Sales | JSON | JSON | 6 435 | 3 | 5 | 05.02.2010 — 26.10.2012 |
| 4 | Amazon Reviews | Неструктурированный | CSV | 568 454 | 4 | 3 | 08.10.1999 — 26.10.2012 |

**Уникальность источников.** Все четыре источника уникальны и не пересекаются по содержанию. Ни один не является продолжением другого. Каждый представляет свой тип данных:
- Retail — транзакции одного интернет-магазина (Великобритания, 2010–2011)
- XML — агрегированная статистика по регионам РФ (2010, 2011)
- JSON — макро-данные розничных продаж США (2010–2012)
- Reviews — тексты отзывов покупателей Amazon (2000–2012)

# Оси OLAP

**Время**. Retail и XML покрывают 2010–2011, Walmart — 2010–2012, Reviews — 2000–2012. Пересечение по времени дают Retail + XML + Walmart. Это основа для проверки гипотезы H1 (временные паттерны).

**География**. Retail — страны, XML — регионы РФ, Walmart — США, Reviews — без геопривязки. Пересечений нет — ось не используется для слияния.

**Товар**. Retail — StockCode/Description, Walmart — только Store, Reviews — ProductId. Прямого пересечения нет — ось остаётся смысловой.

**Клиент**. Retail — Customer ID, Reviews — UserId/ProfileName. XML и Walmart без клиентов. Пересечений нет.

**Итог**. Слияние возможно только по оси «Время» (Retail + XML + Walmart, 2010–2011). Остальные оси — смысловые, для нормализации на следующих этапах.